In [27]:
import pandas as pd
import numpy as np

# https://xgboost.readthedocs.io/en/stable/python/python_intro.html#install-xgboost
import xgboost as xgb
from xgboost import XGBClassifier

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# Read files

In [28]:
# trainingset
X_train = pd.read_csv("/Users/kellyt/Documents/ml/HW2/train1_icu_data.csv")
y_train = pd.read_csv("/Users/kellyt/Documents/ml/HW2/train1_icu_label.csv")

# testset
X_test = pd.read_csv("/Users/kellyt/Documents/ml/HW2/test1_icu_data.csv")
y_test = pd.read_csv("/Users/kellyt/Documents/ml/HW2/test1_icu_label.csv")
# print("x1_train:", x1_train.shape)
print("y1_train:", y_train.shape)

print("x1_test:", X_test.shape)
print("y1_test:", y_test.shape)

# print("training distribution", y1_train.value_counts(normalize=True))
# print("testing distribution", y1_test.value_counts(normalize=True))
# A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().

# y1_train = y1_train.values.ravel()
# y1_test = y1_test.values.ravel()

y1_train: (5000, 1)
x1_test: (1097, 108)
y1_test: (1097, 1)


In [29]:
assert X_train.columns.tolist() == X_test.columns.tolist()
assert set(y_train.iloc[:, 0].unique()).issubset({0, 1})
assert set(y_test.iloc[:, 0].unique()).issubset({0, 1})

print("Feature columns and target labels checked successfully!")

Feature columns and target labels checked successfully!


# Build XGBoost Model

https://xgboost.readthedocs.io/en/stable/python/sklearn_estimator.html

https://www.geeksforgeeks.org/machine-learning/xgbclassifier/

In [30]:
xgb_basic = XGBClassifier(
    objective="binary:logistic",
    random_state=42,
)
xgb_basic

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [31]:
xgb_basic.fit(X_train, y_train)

# Predict class labels
y_pred = xgb_basic.predict(X_test)

# Predict probabilities for the death class
y_prob = xgb_basic.predict_proba(X_test)[:, 1]
y_prob

array([6.8517949e-04, 2.5165966e-02, 7.8643762e-02, ..., 8.2865232e-01,
       9.8505068e-01, 6.8500096e-01], dtype=float32)

In [32]:
print("Accuracy:", round(accuracy_score(y_test, y_pred), 5))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 5))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 5))
print("F1-score:", round(f1_score(y_test, y_pred, zero_division=0), 5))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 5))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.79945
Precision: 0.79041
Recall: 0.81353
F1-score: 0.8018
ROC-AUC: 0.88114

Confusion Matrix:
[[432 118]
 [102 445]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.79      0.80       550
           1       0.79      0.81      0.80       547

    accuracy                           0.80      1097
   macro avg       0.80      0.80      0.80      1097
weighted avg       0.80      0.80      0.80      1097



# XGBoost with hyperparameter tuning

In [38]:
# 1. Five-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 2. Hyperparameter options
param_distributions = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "min_child_weight": [1, 3, 5],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "reg_lambda": [1, 5, 10]
}

# 3. Multiple scoring metrics dictionary
scoring_metrics = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=1
    ),
    param_distributions=param_distributions,
    n_iter=20,
    scoring=scoring_metrics,
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit="f1"  # Refit best model based on F1-score
)

# 4. Search using training data only
xgb_search.fit(X_train, y_train)

# 5. Output Best Parameters & Best Score rounded to 5 decimals
print("Best parameters:")
print(xgb_search.best_params_)

print("\nBest mean CV F1-score:")
print(round(xgb_search.best_score_, 5))

# 6. Create a summary table of all CV results across all metrics
results_df = pd.DataFrame(xgb_search.cv_results_)

# Filter and display relevant columns (parameters + mean test scores for each metric)
param_cols = [f"param_{p}" for p in param_distributions.keys()]
score_cols = [f"mean_test_{m}" for m in scoring_metrics.keys()]

summary_table = results_df[param_cols + score_cols].copy()

# Round all score columns to 5 decimal places for clarity
for col in score_cols:
    summary_table[col] = summary_table[col].round(5)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters:
{'subsample': 1.0, 'reg_lambda': 10, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 1.0}

Best mean CV F1-score:
0.81483


In [39]:
print("\n--- All Cross-Validation Results Table ---")
pd.set_option('display.max_columns', None)
summary_table


--- All Cross-Validation Results Table ---


,param_n_estimators,param_max_depth,param_learning_rate,param_min_child_weight,param_subsample,param_colsample_bytree,param_reg_lambda,mean_test_accuracy,mean_test_precision,mean_test_recall,mean_test_f1,mean_test_roc_auc
0,200,2,0.05,3,0.9,0.9,10,0.8044,0.80621,0.81176,0.80891,0.88335
1,400,2,0.03,5,0.7,0.7,10,0.8050,0.80694,0.81216,0.80945,0.88514
2,200,4,0.10,3,0.9,0.8,1,0.8042,0.80332,0.81647,0.80962,0.88597
3,100,3,0.10,1,1.0,0.8,5,0.8088,0.81207,0.81373,0.81275,0.88655
4,100,5,0.10,3,0.7,1.0,5,0.8036,0.80495,0.81176,0.80828,0.88460
5,200,6,0.10,5,0.9,0.8,10,0.8032,0.80498,0.81098,0.80774,0.88625
6,300,5,0.01,3,0.8,0.9,1,0.8070,0.80734,0.81686,0.81194,0.88526
7,300,5,0.01,1,0.9,0.7,10,0.8052,0.80618,0.81412,0.80999,0.88453
8,100,2,0.05,5,0.9,0.8,10,0.7972,0.79846,0.80627,0.80221,0.87810
9,100,5,0.10,5,0.9,0.8,10,0.8076,0.80855,0.81647,0.81233,0.88728


## Compare with baseline model

In [40]:
scoring_metrics = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

baseline_cv = cross_validate(
    xgb_basic,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring_metrics,
    n_jobs=-1
)

print("Baseline cross-validation results:")

for metric in scoring_metrics:
    scores = baseline_cv[f"test_{metric}"]
    print(f"{metric}: {scores.mean():.4f} (+/- {scores.std():.4f})")

Baseline cross-validation results:
accuracy: 0.7990 (+/- 0.0107)
precision: 0.7991 (+/- 0.0169)
recall: 0.8102 (+/- 0.0183)
f1: 0.8043 (+/- 0.0099)
roc_auc: 0.8832 (+/- 0.0059)


## Evaluate on test set with the best model

In [44]:
# RandomizedSearchCV has already refitted the best model
best_xgb = xgb_search.best_estimator_

y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:, 1]

print("Final Test Results")
print("-------------------")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 5))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 5))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 5))
print("F1-score:", round(f1_score(y_test, y_pred, zero_division=0), 5))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 5))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Final Test Results
-------------------
Accuracy: 0.79125
Precision: 0.77993
Recall: 0.80987
F1-score: 0.79462
ROC-AUC: 0.88145

Confusion Matrix:
[[425 125]
 [104 443]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.77      0.79       550
           1       0.78      0.81      0.79       547

    accuracy                           0.79      1097
   macro avg       0.79      0.79      0.79      1097
weighted avg       0.79      0.79      0.79      1097

